# Pipeline Initialization & Environment Guard

This cell performs all pre-processing setup before reading large-scale
cluster usage logs (~5GB):

1. Verifies that the notebook is executed in the correct conda environment("stat427") to ensure reproducibility and dependency consistency.

2. Automatically detects the project root directory (supports running either from project root or a subfolder like /notebooks).

3. Defines all key input/output paths in a centralized manner.

4. Sets global constants for chunked processing (CHUNK_SIZE), hashing buckets (BUCKET_COUNT), and reproducibility (RANDOM_SEED).

5. Creates required output directories if they do not already exist.

6. Confirms that the raw merged cluster usage file is present before proceeding to heavy data processing.


In [1]:
# load necessary libraries and set up environment
import os
import re
import sys
import gzip
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

# Check environment
print("Python executable:", sys.executable)
print("CONDA_DEFAULT_ENV:", os.environ.get("CONDA_DEFAULT_ENV"))
assert os.environ.get("CONDA_DEFAULT_ENV") == "stat427",     "ERROR: Please run this notebook in the stat427 environment created from environment.yml." # Check that we're in the right conda environment, which ensures we have the correct package versions installed.

# Determine project root
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "environment.yml").exists() and (PROJECT_ROOT.parent / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent # Move up one directory if environment.yml is not found in the current directory but is found in the parent directory. This allows the notebook to be run from either the root or a subdirectory of the project.


# Define paths and constants
INPUT_GZ = PROJECT_ROOT / "iccp-cluster-usage-analysis-selected" / "analysis_ready" / "whole_cluster_usage_merged_raw.txt.gz"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
CHUNK_SIZE = 200_000
BUCKET_COUNT = 256
RANDOM_SEED = 427

RAW_VALID_PATH = OUTPUT_DIR / "master_2025_raw_valid.csv.gz"
SUBMISSION_PATH = OUTPUT_DIR / "master_2025_joblevel_submission.csv.gz"
TERMINAL_PATH = OUTPUT_DIR / "master_2025_joblevel_terminal.csv.gz"
TMP_BUCKET_DIR = OUTPUT_DIR / "_tmp_joblevel_buckets"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

if not INPUT_GZ.exists():
    raise FileNotFoundError(f"Input file not found: {INPUT_GZ}") # Check that the input file exists before proceeding.

print(f"INPUT_GZ: {INPUT_GZ}")
print(f"PROJECT_ROOT: {PROJECT_ROOT.resolve()}")
print(f"OUTPUT_DIR: {OUTPUT_DIR.resolve()}")
print(f"CHUNK_SIZE: {CHUNK_SIZE:,}")


Python executable: /opt/anaconda3/envs/stat427/bin/python
CONDA_DEFAULT_ENV: stat427
INPUT_GZ: /Volumes/Store/427/iccp-cluster-usage-analysis-selected/analysis_ready/whole_cluster_usage_merged_raw.txt.gz
PROJECT_ROOT: /Volumes/Store/427
OUTPUT_DIR: /Volumes/Store/427/outputs
CHUNK_SIZE: 200,000


/opt/anaconda3/envs/stat427/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Step 1: Validate file schema (header-only read)

To avoid loading the full ~5GB dataset into memory, we read only the header (nrows=0) to inspect the column structure.

This ensures:
- The file is correctly parsed using '|' as separator.
- The expected number of columns (120) is present.
- No structural changes occurred in the raw merged dataset.

This acts as a lightweight schema validation step before chunk-based processing begins.


In [ ]:
# Read the header of the input file to get column names and verify the expected number of columns. 
header_df = pd.read_csv(
    INPUT_GZ,
    sep='|',
    compression='gzip',
    engine='python',
    nrows=0,
)

header_cols = header_df.columns.tolist()
print(f"Header column count: {len(header_cols)}")
assert len(header_cols) == 120, f"Expected 120 columns, got {len(header_cols)}" # Verify that the number of columns matches the expected count, which helps catch issues with file formatting or reading.

print("First 10 columns:", header_cols[:10])
print("Last 10 columns:", header_cols[-10:])


Header column count: 120
First 10 columns: ['Account', 'AdminComment', 'AllocCPUS', 'AllocNodes', 'AllocTRES', 'AssocID', 'AveCPU', 'AveCPUFreq', 'AveDiskRead', 'AveDiskWrite']
Last 10 columns: ['TRESUsageOutMin', 'TRESUsageOutMinNode', 'TRESUsageOutMinTask', 'TRESUsageOutTot', 'UID', 'User', 'UserCPU', 'WCKey', 'WCKeyID', 'WorkDir']


## Header Check Passed

- Column count: **120 (as expected)**
- Delimiter parsing is correct
- Schema appears structurally consistent

The dataset is ready for chunk-based ingestion.

# Cell: Build `master_2025_raw_valid.csv.gz` (chunked ingest + minimal cleaning)

This cell creates a **job-level “raw but usable” dataset** for year 2025 from the merged whole-cluster SLURM log.

What it does:

1. **Define missing-value normalization rules**
   - Treats `""`, `unknown`, `none`, `n/a` (case-insensitive) as missing.

2. **Parse requested memory (`ReqMem`) into a unified numeric unit**
   - Uses a regex to extract `(value, unit)` from strings like `500M`, `2G`, `1.5T`, etc.
   - Converts everything to **MB** in a new column `ReqMem_MB`.

3. **Ensure reproducibility / rerun-safety**
   - Deletes any existing output files (`RAW_VALID_PATH`, `SUBMISSION_PATH`, `TERMINAL_PATH`)
   - Removes any temporary bucket directory (`TMP_BUCKET_DIR`) so rerunning does not append duplicates.

4. **Estimate input line count**
   - Counts total data lines in the gz file (excluding header) for later “bad-line skip” estimation.

5. **Chunked reading to avoid memory issues**
   - Reads the gz file with `chunksize=CHUNK_SIZE` and `dtype='string'`.
   - Skips malformed rows via `on_bad_lines='skip'` (keeps the pipeline running on large logs).

6. **Per-chunk cleaning + feature derivation**
   - Normalizes missing tokens in key identifier/time/resource columns.
   - Filters to jobs with `Submit` starting with `"2025"`.
   - Simplifies `State` to the first token (e.g., `"CANCELLED by ..."` → `"CANCELLED"`).
   - Computes:
     - `WaitTimeSec = Start - Submit`
     - `RunTimeSec  = End - Start`

7. **Write standardized output**
   - Writes only the selected columns (`RAW_OUTPUT_COLUMNS`) to a gzipped CSV:
     - `outputs/master_2025_raw_valid.csv.gz`
   - Appends chunk-by-chunk; header is written only once.

8. **Report ingestion summary**
   - `parsed_rows`: rows successfully parsed by pandas (after skipping malformed lines)
   - `written_rows`: rows written to output (after 2025 filter)
   - `estimated_bad_lines_skipped`: rough estimate of skipped malformed rows
   - `estimated_rows_dropped_by_2025_filter`: parsed but not in year 2025



In [ ]:
# Define functions for data normalization and parsing, as well as a function to count lines in the input file for estimating bad lines.
NA_TOKENS = {"", "unknown", "none", "n/a"}
REQMEM_PATTERN = re.compile(r"^\s*([0-9]*\.?[0-9]+)\s*([kKmMgGtT])(?:[cCnN])?\s*$")

RAW_OUTPUT_COLUMNS = [
    "JobIDRaw", "JobID", "JobName",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit", "Eligible", "Start", "End",
    "State", "Reason", "ExitCode",
    "ReqCPUS", "ReqMem", "ReqMem_MB", "ReqNodes",
    "AllocCPUS", "AllocNodes", "AllocTRES",
    "ElapsedRaw", "TimelimitRaw",
    "WaitTimeSec", "RunTimeSec",
]

for p in [RAW_VALID_PATH, SUBMISSION_PATH, TERMINAL_PATH]:
    if p.exists():
        p.unlink()
if TMP_BUCKET_DIR.exists():
    shutil.rmtree(TMP_BUCKET_DIR)


def normalize_missing(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    return s.mask(s.str.lower().isin(NA_TOKENS))


def parse_reqmem_mb(series: pd.Series) -> pd.Series:
    s = normalize_missing(series)
    extracted = s.str.extract(REQMEM_PATTERN)
    numeric = pd.to_numeric(extracted[0], errors='coerce')
    unit = extracted[1].str.upper()
    factor = unit.map({"K": 1/1024, "M": 1, "G": 1024, "T": 1024*1024})
    return numeric * factor


def count_input_data_lines(path: Path) -> int:
    with gzip.open(path, mode='rt', encoding='utf-8', errors='replace') as f:
        _ = next(f, None)  # header
        return sum(1 for _ in f)

print("Counting input lines (for bad-line estimation)...")
input_data_lines = count_input_data_lines(INPUT_GZ)
print(f"Input data lines (without header): {input_data_lines:,}")

parsed_rows = 0
written_rows = 0
header_written = False

normalize_cols = [
    "JobIDRaw", "JobID", "JobName", "User", "Group", "Account",
    "Partition", "QOS", "Submit", "Eligible", "Start", "End", "State",
    "Reason", "ExitCode", "ReqCPUS", "ReqMem", "ReqNodes",
    "AllocCPUS", "AllocNodes", "AllocTRES", "ElapsedRaw", "TimelimitRaw",
]

reader = pd.read_csv(
    INPUT_GZ,
    sep='|',
    compression='gzip',
    engine='python',
    chunksize=CHUNK_SIZE,
    on_bad_lines='skip',
    dtype='string',
)

for chunk_idx, chunk in enumerate(tqdm(reader, desc="Building raw_valid"), start=1):
    parsed_rows += len(chunk)

    for c in normalize_cols:
        if c in chunk.columns:
            chunk[c] = normalize_missing(chunk[c])

    chunk = chunk[chunk["Submit"].str[:4].eq("2025").fillna(False)].copy()
    if chunk.empty:
        continue

    chunk["State"] = normalize_missing(chunk["State"]).str.split().str[0]
    chunk["ReqMem_MB"] = parse_reqmem_mb(chunk["ReqMem"])

    submit_ts = pd.to_datetime(chunk["Submit"], errors='coerce')
    start_ts = pd.to_datetime(chunk["Start"], errors='coerce')
    end_ts = pd.to_datetime(chunk["End"], errors='coerce')

    chunk["WaitTimeSec"] = (start_ts - submit_ts).dt.total_seconds()
    chunk["RunTimeSec"] = (end_ts - start_ts).dt.total_seconds()

    out_chunk = chunk.reindex(columns=RAW_OUTPUT_COLUMNS)
    out_chunk.to_csv(
        RAW_VALID_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    written_rows += len(out_chunk)

    if chunk_idx % 20 == 0:
        print(f"Chunk {chunk_idx}: parsed_rows={parsed_rows:,}, written_rows={written_rows:,}")

bad_line_estimate = max(input_data_lines - parsed_rows, 0)
filter_drop_estimate = parsed_rows - written_rows

print("\nCell 2 summary")
print(f"parsed_rows (after parser, bad lines skipped): {parsed_rows:,}")
print(f"written_rows (2025 raw_valid): {written_rows:,}")
print(f"estimated_bad_lines_skipped: {bad_line_estimate:,}")
print(f"estimated_rows_dropped_by_2025_filter: {filter_drop_estimate:,}")
print(f"raw_valid output: {RAW_VALID_PATH}")


Counting input lines (for bad-line estimation)...
Input data lines (without header): 6,123,190


Building raw_valid: 20it [02:35,  7.31s/it]

Chunk 20: parsed_rows=3,989,167, written_rows=3,960,600


Building raw_valid: 31it [03:56,  7.64s/it]


Cell 2 summary
parsed_rows (after parser, bad lines skipped): 6,102,325
written_rows (2025 raw_valid): 6,059,140
estimated_bad_lines_skipped: 20,865
estimated_rows_dropped_by_2025_filter: 43,185
raw_valid output: /Volumes/Store/427/outputs/master_2025_raw_valid.csv.gz


Result: a consistent 2025-only dataset with unified schema and derived timing features,ready for downstream job-level aggregation and analysis.

# Quality report on raw_valid

In [ ]:
state_target = ["COMPLETED", "FAILED", "CANCELLED", "TIMEOUT", "OOM", "PENDING", "OTHER"]
state_counts = {k: 0 for k in state_target}

raw_rows = 0
start_missing = 0
end_missing = 0
wait_calc = 0
runtime_calc = 0
submit_min = None
submit_max = None

reader = pd.read_csv(
    RAW_VALID_PATH,
    compression='gzip',
    chunksize=CHUNK_SIZE,
)

for chunk in tqdm(reader, desc="Quality stats"):
    raw_rows += len(chunk)

    submit_ts = pd.to_datetime(chunk["Submit"], errors='coerce')
    chunk_min = submit_ts.min()
    chunk_max = submit_ts.max()
    if pd.notna(chunk_min):
        submit_min = chunk_min if submit_min is None else min(submit_min, chunk_min)
    if pd.notna(chunk_max):
        submit_max = chunk_max if submit_max is None else max(submit_max, chunk_max)

    state_series = chunk["State"].astype("string").str.strip().str.upper()
    state_series = state_series.mask(state_series == "OUT_OF_MEMORY", "OOM")
    state_series = state_series.where(state_series.isin(state_target[:-1]), "OTHER")
    vc = state_series.value_counts(dropna=False)
    for k, v in vc.items():
        state_counts[k] = state_counts.get(k, 0) + int(v)

    start_missing += int(chunk["Start"].isna().sum())
    end_missing += int(chunk["End"].isna().sum())
    wait_calc += int(pd.to_numeric(chunk["WaitTimeSec"], errors='coerce').notna().sum())
    runtime_calc += int(pd.to_numeric(chunk["RunTimeSec"], errors='coerce').notna().sum())

print("raw_valid_total_rows:", raw_rows)
print("Submit min:", submit_min)
print("Submit max:", submit_max)
print("\nState distribution:")
for k in state_target:
    print(f"  {k}: {state_counts.get(k, 0):,}")

print("\nMissingness:")
print(f"  Start missing rate: {start_missing/raw_rows:.4%}")
print(f"  End missing rate:   {end_missing/raw_rows:.4%}")

print("\nComputable ratios:")
print(f"  WaitTimeSec computable ratio: {wait_calc/raw_rows:.4%}")
print(f"  RunTimeSec computable ratio:  {runtime_calc/raw_rows:.4%}")


Quality stats: 13it [00:07,  1.50it/s]/opt/anaconda3/envs/stat427/lib/python3.12/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
Quality stats: 17it [00:09,  1.64it/s]/opt/anaconda3/envs/stat427/lib/python3.12/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (1,13) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
Quality stats: 31it [00:16,  1.88it/s]

raw_valid_total_rows: 6059140
Submit min: 2025-01-01 13:01:57
Submit max: 2025-12-30 23:58:39

State distribution:
  COMPLETED: 4,659,043
  FAILED: 586,508
  CANCELLED: 641,040
  TIMEOUT: 122,388
  OOM: 43,526
  PENDING: 6,219
  OTHER: 416

Missingness:
  Start missing rate: 8.5026%
  End missing rate:   0.1038%

Computable ratios:
  WaitTimeSec computable ratio: 91.4974%
  RunTimeSec computable ratio:  91.4962%


# Build master_2025_joblevel_submission (hash buckets + per-bucket dedup)

In [ ]:
TMP_BUCKET_DIR.mkdir(parents=True, exist_ok=True)
for p in TMP_BUCKET_DIR.glob("bucket_*.csv.gz"):
    p.unlink()

if SUBMISSION_PATH.exists():
    SUBMISSION_PATH.unlink()

joblevel_work_cols = [
    "JobIDRaw", "JobID", "JobName",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit", "Eligible", "Start", "End",
    "State", "Reason", "ExitCode",
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
    "AllocCPUS", "AllocNodes", "AllocTRES",
    "ElapsedRaw", "TimelimitRaw",
    "WaitTimeSec", "RunTimeSec",
]

submission_cols = [
    "JobIDRaw",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
    "State",
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
    "AllocCPUS", "AllocNodes",
]

row_cursor = 0
bucket_rows = 0

reader = pd.read_csv(
    RAW_VALID_PATH,
    compression='gzip',
    chunksize=CHUNK_SIZE,
)

for chunk in tqdm(reader, desc="Bucketizing for job-level"):
    chunk = chunk.reindex(columns=joblevel_work_cols).copy()
    chunk = chunk[chunk["JobIDRaw"].notna()].copy()
    chunk["JobIDRaw"] = chunk["JobIDRaw"].astype("string").str.strip()
    chunk = chunk[chunk["JobIDRaw"].ne("")].copy()

    if chunk.empty:
        continue

    n = len(chunk)
    chunk["_row_order"] = np.arange(row_cursor, row_cursor + n, dtype=np.int64)
    row_cursor += n

    bucket_ids = (
        pd.util.hash_pandas_object(chunk["JobIDRaw"], index=False).astype("uint64") % BUCKET_COUNT
    ).astype("int64")
    chunk["_bucket"] = bucket_ids.values

    for bucket_id, sub in chunk.groupby("_bucket", sort=False):
        path = TMP_BUCKET_DIR / f"bucket_{int(bucket_id):03d}.csv.gz"
        sub.drop(columns=["_bucket"]).to_csv(
            path,
            mode='a',
            index=False,
            header=not path.exists(),
            compression='gzip',
        )
        bucket_rows += len(sub)

print(f"Bucketized rows: {bucket_rows:,}")
print(f"Bucket directory: {TMP_BUCKET_DIR}")

header_written = False
submission_jobs = 0
bucket_files = sorted(TMP_BUCKET_DIR.glob("bucket_*.csv.gz"))

for path in tqdm(bucket_files, desc="Dedup submission per bucket"):
    bdf = pd.read_csv(path, compression='gzip')
    if bdf.empty:
        continue

    bdf["_row_order"] = pd.to_numeric(bdf["_row_order"], errors='coerce').fillna(10**18)
    bdf["_SubmitTS"] = pd.to_datetime(bdf["Submit"], errors='coerce')
    bdf["_SubmitTS_sort"] = bdf["_SubmitTS"].fillna(pd.Timestamp.max)

    bdf = bdf.sort_values(
        ["JobIDRaw", "_SubmitTS_sort", "_row_order"],
        ascending=[True, True, True],
        kind="mergesort",
    )

    selected = bdf.drop_duplicates(subset=["JobIDRaw"], keep="first")
    out = selected.reindex(columns=submission_cols)

    out.to_csv(
        SUBMISSION_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    submission_jobs += len(out)

print(f"submission unique jobs: {submission_jobs:,}")
print(f"submission output: {SUBMISSION_PATH}")


Bucketizing for job-level: 13it [00:39,  2.98s/it]/opt/anaconda3/envs/stat427/lib/python3.12/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
Bucketizing for job-level: 17it [00:52,  3.00s/it]/opt/anaconda3/envs/stat427/lib/python3.12/site-packages/tqdm/std.py:1181: DtypeWarning: Columns (1,13) have mixed types. Specify dtype option on import or set low_memory=False.
  for obj in iterable:
Bucketizing for job-level: 31it [01:30,  2.91s/it]


Bucketized rows: 6,059,140
Bucket directory: /Volumes/Store/427/outputs/_tmp_joblevel_buckets


Dedup submission per bucket: 100%|██████████| 256/256 [00:41<00:00,  6.19it/s]

submission unique jobs: 6,025,326
submission output: /Volumes/Store/427/outputs/master_2025_joblevel_submission.csv.gz


# Build master_2025_joblevel_terminal (hash buckets + terminal rule)

In [ ]:
if TERMINAL_PATH.exists():
    TERMINAL_PATH.unlink()

terminal_cols = [
    "JobIDRaw",
    "User", "Group", "Account",
    "Partition", "QOS",
    "Submit",
    "State",
    "ReqCPUS", "ReqMem_MB", "ReqNodes",
    "AllocCPUS", "AllocNodes",
    "Start", "End", "ElapsedRaw", "ExitCode", "Reason",
    "WaitTimeSec", "RunTimeSec",
]

header_written = False
terminal_jobs = 0
bucket_files = sorted(TMP_BUCKET_DIR.glob("bucket_*.csv.gz"))

for path in tqdm(bucket_files, desc="Dedup terminal per bucket"):
    bdf = pd.read_csv(path, compression='gzip')
    if bdf.empty:
        continue

    bdf["_row_order"] = pd.to_numeric(bdf["_row_order"], errors='coerce').fillna(10**18)

    bdf["_SubmitTS"] = pd.to_datetime(bdf["Submit"], errors='coerce')
    bdf["_StartTS"] = pd.to_datetime(bdf["Start"], errors='coerce')
    bdf["_EndTS"] = pd.to_datetime(bdf["End"], errors='coerce')

    # priority: 2 (End exists), 1 (Start exists only), 0 (Submit only)
    bdf["_priority"] = np.select(
        [bdf["_EndTS"].notna(), bdf["_StartTS"].notna()],
        [2, 1],
        default=0,
    )

    bdf["_TimeKey"] = bdf["_EndTS"].where(
        bdf["_EndTS"].notna(),
        bdf["_StartTS"].where(bdf["_StartTS"].notna(), bdf["_SubmitTS"]),
    )
    bdf["_TimeKey_sort"] = bdf["_TimeKey"].fillna(pd.Timestamp("1900-01-01"))

    bdf = bdf.sort_values(
        ["JobIDRaw", "_priority", "_TimeKey_sort", "_row_order"],
        ascending=[True, False, False, True],
        kind="mergesort",
    )

    selected = bdf.drop_duplicates(subset=["JobIDRaw"], keep="first")
    out = selected.reindex(columns=terminal_cols)

    out.to_csv(
        TERMINAL_PATH,
        mode='a',
        index=False,
        header=not header_written,
        compression='gzip',
    )
    header_written = True
    terminal_jobs += len(out)

print(f"terminal unique jobs: {terminal_jobs:,}")
print(f"terminal output: {TERMINAL_PATH}")


Dedup terminal per bucket: 100%|██████████| 256/256 [01:10<00:00,  3.64it/s]

terminal unique jobs: 6,025,326
terminal output: /Volumes/Store/427/outputs/master_2025_joblevel_terminal.csv.gz


# Final sanity checks

In [7]:
ID_TMP_DIR = OUTPUT_DIR / "_tmp_id_compare"
if ID_TMP_DIR.exists():
    shutil.rmtree(ID_TMP_DIR)
ID_TMP_DIR.mkdir(parents=True, exist_ok=True)


def bucketize_jobids(path: Path, prefix: str, chunksize: int = CHUNK_SIZE, bucket_count: int = BUCKET_COUNT):
    for chunk in pd.read_csv(path, compression='gzip', usecols=["JobIDRaw"], chunksize=chunksize):
        chunk = chunk[chunk["JobIDRaw"].notna()].copy()
        chunk["JobIDRaw"] = chunk["JobIDRaw"].astype("string").str.strip()
        chunk = chunk[chunk["JobIDRaw"].ne("")]
        if chunk.empty:
            continue

        bucket_ids = (
            pd.util.hash_pandas_object(chunk["JobIDRaw"], index=False).astype("uint64") % bucket_count
        ).astype("int64")
        chunk["_bucket"] = bucket_ids.values

        for bid, sub in chunk.groupby("_bucket", sort=False):
            out_path = ID_TMP_DIR / f"{prefix}_{int(bid):03d}.csv.gz"
            sub[["JobIDRaw"]].to_csv(
                out_path,
                mode='a',
                index=False,
                header=not out_path.exists(),
                compression='gzip',
            )


def read_id_set(path: Path) -> set:
    if not path.exists():
        return set()
    df = pd.read_csv(path, compression='gzip')
    if df.empty:
        return set()
    return set(df["JobIDRaw"].dropna().astype(str).str.strip().tolist())


def reservoir_sample_jobids(path: Path, k: int = 5, seed: int = RANDOM_SEED):
    rng = random.Random(seed)
    sample = []
    seen = 0
    for chunk in pd.read_csv(path, compression='gzip', usecols=["JobIDRaw"], chunksize=CHUNK_SIZE):
        for jid in chunk["JobIDRaw"].dropna().astype(str):
            jid = jid.strip()
            if not jid:
                continue
            seen += 1
            if len(sample) < k:
                sample.append(jid)
            else:
                j = rng.randint(1, seen)
                if j <= k:
                    sample[j - 1] = jid
    return sample


def fetch_rows_for_job(path: Path, job_id: str, columns: list, max_rows: int = 50):
    chunks = []
    for chunk in pd.read_csv(path, compression='gzip', usecols=columns, chunksize=CHUNK_SIZE):
        sub = chunk[chunk["JobIDRaw"].astype(str) == job_id]
        if not sub.empty:
            chunks.append(sub)
    if not chunks:
        return pd.DataFrame(columns=columns)
    out = pd.concat(chunks, ignore_index=True)
    return out.head(max_rows)


print("Bucketing JobIDRaw for set-difference sanity check...")
bucketize_jobids(SUBMISSION_PATH, "sub")
bucketize_jobids(TERMINAL_PATH, "ter")

submission_jobs = 0
terminal_jobs = 0
submission_dup_within_output = 0
terminal_dup_within_output = 0
submission_not_terminal = 0
terminal_not_submission = 0

for i in range(BUCKET_COUNT):
    sub_ids = read_id_set(ID_TMP_DIR / f"sub_{i:03d}.csv.gz")
    ter_ids = read_id_set(ID_TMP_DIR / f"ter_{i:03d}.csv.gz")

    submission_jobs += len(sub_ids)
    terminal_jobs += len(ter_ids)

    submission_not_terminal += len(sub_ids - ter_ids)
    terminal_not_submission += len(ter_ids - sub_ids)

print("\nFinal counts")
print(f"submission job count: {submission_jobs:,}")
print(f"terminal job count:   {terminal_jobs:,}")
print(f"submission_only JobIDRaw count: {submission_not_terminal:,}")
print(f"terminal_only JobIDRaw count:   {terminal_not_submission:,}")

sample_jobids = reservoir_sample_jobids(SUBMISSION_PATH, k=5, seed=RANDOM_SEED)
print("\nSample JobIDRaw for manual audit:", sample_jobids)

RAW_SNAPSHOT_COLS = [
    "JobIDRaw", "JobID", "JobName", "Submit", "Eligible", "Start", "End",
    "State", "Reason", "ExitCode", "ReqCPUS", "ReqMem_MB", "ReqNodes",
    "AllocCPUS", "AllocNodes", "ElapsedRaw", "WaitTimeSec", "RunTimeSec",
]

SUBMISSION_COLS = [
    "JobIDRaw", "User", "Group", "Account", "Partition", "QOS",
    "Submit", "State", "ReqCPUS", "ReqMem_MB", "ReqNodes", "AllocCPUS", "AllocNodes",
]

TERMINAL_COLS = [
    "JobIDRaw", "User", "Group", "Account", "Partition", "QOS",
    "Submit", "State", "ReqCPUS", "ReqMem_MB", "ReqNodes", "AllocCPUS", "AllocNodes",
    "Start", "End", "ElapsedRaw", "ExitCode", "Reason", "WaitTimeSec", "RunTimeSec",
]

for jid in sample_jobids:
    print("\n" + "=" * 100)
    print(f"JobIDRaw = {jid}")

    raw_rows = fetch_rows_for_job(RAW_VALID_PATH, jid, RAW_SNAPSHOT_COLS, max_rows=20)
    sub_row = fetch_rows_for_job(SUBMISSION_PATH, jid, SUBMISSION_COLS, max_rows=5)
    ter_row = fetch_rows_for_job(TERMINAL_PATH, jid, TERMINAL_COLS, max_rows=5)

    print("raw_valid snapshots (up to 20 rows):")
    display(raw_rows)

    print("selected submission row:")
    display(sub_row)

    print("selected terminal row:")
    display(ter_row)


Bucketing JobIDRaw for set-difference sanity check...

Final counts
submission job count: 6,025,326
terminal job count:   6,025,326
submission_only JobIDRaw count: 0
terminal_only JobIDRaw count:   0

Sample JobIDRaw for manual audit: ['3863146', '239502', '4046351', '4450131', '246249']

JobIDRaw = 3863146


/var/folders/_k/0kyfgdr93hd8vb42rdk_j0040000gn/T/ipykernel_9910/1926827800.py:62: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, compression='gzip', usecols=columns, chunksize=CHUNK_SIZE):
/var/folders/_k/0kyfgdr93hd8vb42rdk_j0040000gn/T/ipykernel_9910/1926827800.py:62: DtypeWarning: Columns (1,13) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(path, compression='gzip', usecols=columns, chunksize=CHUNK_SIZE):


raw_valid snapshots (up to 20 rows):


,JobIDRaw,JobID,JobName,Submit,Eligible,Start,End,State,Reason,ExitCode,ReqCPUS,ReqMem_MB,ReqNodes,AllocCPUS,AllocNodes,ElapsedRaw,WaitTimeSec,RunTimeSec
0,3863146,3863146,data_leroux_parsimonious_identifiable_590_87_TRUE,2025-07-30T16:29:58,2025-07-30T16:29:58,NaN,2025-07-30T16:42:49,CANCELLED,NaN,0:0,4,16384.0,1,0,0,0,NaN,NaN


selected submission row:


...

,JobIDRaw,JobID,JobName,Submit,Eligible,Start,End,State,Reason,ExitCode,ReqCPUS,ReqMem_MB,ReqNodes,AllocCPUS,AllocNodes,ElapsedRaw,WaitTimeSec,RunTimeSec
0,246249,245962_251,rbim,2025-02-05T08:52:59,2025-02-05T08:52:59,2025-02-05T09:02:57,2025-02-05T09:04:31,COMPLETED,AssocMaxJobsLimit,0:0,8,16000.0,1,8,1,94,598.0,94.0


selected submission row:


,JobIDRaw,User,Group,Account,Partition,QOS,Submit,State,ReqCPUS,ReqMem_MB,ReqNodes,AllocCPUS,AllocNodes
0,246249,mwliu2,CampusClusterUsers,campusclusterusers,secondary,normal,2025-02-05T08:52:59,COMPLETED,8,16000.0,1,8,1


selected terminal row:


,JobIDRaw,User,Group,Account,Partition,QOS,Submit,State,ReqCPUS,ReqMem_MB,ReqNodes,AllocCPUS,AllocNodes,Start,End,ElapsedRaw,ExitCode,Reason,WaitTimeSec,RunTimeSec
0,246249,mwliu2,CampusClusterUsers,campusclusterusers,secondary,normal,2025-02-05T08:52:59,COMPLETED,8,16000.0,1,8,1,2025-02-05T09:02:57,2025-02-05T09:04:31,94,0:0,AssocMaxJobsLimit,598.0,94.0
